# LoRA Experiments

<img src="./imgs/lora.png" width="1000">

Essentially LoRA apply the low rank update for the weight matrices of the model,

$$\begin{aligned}
W &= W + \Delta W  \\
  &= W + A \cdot B,
\end{aligned}$$

where we assume both $A$ and $B$ are low rank matrices of rank $r$, a smaller $r$ leads to less parameters for fine-tuning. To control the scaling, we introduce another hyperparameter $\alpha$ that determines the magnitude of the weight changes introduced LoRA layer,
$$\begin{aligned}
W &= W + \alpha \cdot \Delta W  \\
  &= W + \alpha \cdot A \cdot B,
\end{aligned}$$
a higher $\alpha$ leads to more aggressive fine-tuning, while a lower $\alpha$ leads to more subtle weight updates.

Furthermore, we initialize $A$ from a normal distribution with the standard deviation determined by the square root of the rank $r$ to control the initial values of $A$ to not be too large, and $B$ is initialized to zeros. The reason for this is that at the beginning of the fine-tuning training, we do not want the weight updates introduced by LoRA to impact the original model weights, i.e., $AB = 0$ as $B = 0$. We often apply LoRA to the feedforward (linear) layers of the model, for instance, given the forward pass

```python
def forward(self, x):
    x = self.linear1(x)
    x = F.relu(x)
    x = self.linear2(x)

    return x
```
we add the LoRA layers as follows

```python
def forward(self, x):
    x = self.linear1(x) + self.lora1(x)
    x = F.relu(x)
    x = self.linear2(x) + self.lora2(x)

    return x
```

In [1]:
import torch
from torch import nn
%load_ext watermark
%watermark --conda -p torch,transformers,datasets,lightning
from .autonotebook import tqdm as notebook_tqdm

/home/helloimlixin/anaconda3/envs/research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch       : 2.4.1
transformers: 4.48.2
datasets    : not installed
lightning   : 2.5.0.post0

conda environment: research



In [2]:
class LoRALayer(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha):
        """
        lora layer implementation, which is usually applied to a neural network's linear (feedforward) layers.
        :param in_features: input dimension of the layer we are going to apply LoRA
        :param out_features: the respective output dimension of the layer we are going to apply LoRA
        :param rank: a hyperparameter that determines the rank of the low-rank matrices used for adaptation
        :param alpha: a hyperparameter that determines the magnitude of the weight changes introduced LoRA layer
        """
        super().__init__()

        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self._A = nn.Parameter(torch.randn(in_features, rank) * std_dev)
        self._B = nn.Parameter(torch.zeros(rank, out_features))
        self._alpha = alpha

    def forward(self, x):
        x = self._alpha * (x @ self._A @ self._B)
        return x

In [3]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank, alpha):
        """
        lora linear layer implementation, which is usually applied to a neural network's linear (feedforward) layers.
        :param linear: the linear layer we are going to apply LoRA
        :param rank: a hyperparameter that determines the rank of the low-rank matrices used for adaptation
        :param alpha: a hyperparameter that determines the magnitude of the weight changes introduced LoRA layer
        """
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

# Testing the LoRA Layer

In [4]:
torch.manual_seed(123)

in_features = 10
out_features = 1
rank = 2

linear_layer = nn.Linear(in_features, out_features).requires_grad_(False)  # freeze the original linear layer

In [5]:
# print the weight matrix of the linear layer
weight_matrix = linear_layer.weight
print(weight_matrix.size())  # torch.Size([1, 10])
print(weight_matrix)

torch.Size([1, 10])
Parameter containing:
tensor([[-0.1290,  0.0105, -0.1571,  0.1193, -0.2694,  0.2318, -0.2298, -0.2514,
         -0.1998,  0.1432]])


In [6]:
inputs = torch.randn(1, in_features)

# test the linear layer forward pass
y = linear_layer(inputs)  # x @ W.T + b
print(y.size())
print(y)

torch.Size([1, 1])
tensor([[0.4634]])


In [7]:
# use the LoRALinear layer to apply LoRA to the linear layer
lora_layer = LoRALinear(linear_layer, rank, alpha=1)

# test the LoRALinear layer forward pass
y = lora_layer(inputs)
print(y.size())
print(y)

torch.Size([1, 1])
tensor([[0.4634]], grad_fn=<AddBackward0>)


Since $B = 0$, the output of the LoRALinear layer is the same as the original linear layer. The LoRA layer will only start to alter the network weights when training starts.

In [8]:
# a simulated weight update for the LoRA layer
lora_layer.lora._B = nn.Parameter(lora_layer.lora._B + 0.01 * inputs[0])   # update B with the first input sampled

In [9]:
# now the output of the LoRALinear layer is different from the original linear layer
y = lora_layer(inputs)
print(y.size())
print(y)

torch.Size([1, 10])
tensor([[0.4416, 0.4638, 0.4686, 0.4860, 0.4575, 0.4903, 0.4718, 0.4533, 0.4725,
         0.4522]], grad_fn=<AddBackward0>)


# A Hands-On Example: Fine-Tuning DistilBERT with LoRA

In [10]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
for param in model.parameters():
    param.requires_grad = False

In [13]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [14]:
from functools import partial

# define hyperparameters for LoRA
lora_rank = 8
lora_alpha = 16
lora_dropout = 0.05
lora_query = True
lora_key = False
lora_value = True
lora_projection = False
lora_mlp = False
lora_head = False

In [15]:
layers = []

lora_assign = partial(LoRALinear, rank=lora_rank, alpha=lora_alpha)

In [16]:
for layer in model.distilbert.transformer.layer:
    if lora_query:
        layer.attention.q_lin = lora_assign(layer.attention.q_lin)
    if lora_key:
        layer.attention.k_lin = lora_assign(layer.attention.k_lin)
    if lora_value:
        layer.attention.v_lin = lora_assign(layer.attention.v_lin)
    if lora_projection:
        layer.attention.out_lin = lora_assign(layer.attention.out_lin)
    if lora_mlp:
        layer.ffn.lin1 = lora_assign(layer.ffn.lin1)
        layer.ffn.lin2 = lora_assign(layer.ffn.lin2)

if lora_head:
    model.pre_classifier = lora_assign(model.pre_classifier)
    model.classifier = lora_assign(model.classifier)

In [17]:
# inspect the model after applying LoRA
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): LoRALinear(
              (linear): Linear(in_features=768, out_features=768, bias=True)
              (lora): LoRALayer()
            )
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): LoRALinear(
              (linear): Linear(in_features=768, out_features=768, bias=True)
              (lora): LoRALayer()
            )
            (out_lin): Linear(in_features=768, out_features=768, bias=True)


In [18]:
# check if layers are frozen correctly
for name, param in model.named_parameters():
    print(f"{name} gradient enabled: {param.requires_grad}")

distilbert.embeddings.word_embeddings.weight gradient enabled: False
distilbert.embeddings.position_embeddings.weight gradient enabled: False
distilbert.embeddings.LayerNorm.weight gradient enabled: False
distilbert.embeddings.LayerNorm.bias gradient enabled: False
distilbert.transformer.layer.0.attention.q_lin.linear.weight gradient enabled: False
distilbert.transformer.layer.0.attention.q_lin.linear.bias gradient enabled: False
distilbert.transformer.layer.0.attention.q_lin.lora._A gradient enabled: True
distilbert.transformer.layer.0.attention.q_lin.lora._B gradient enabled: True
distilbert.transformer.layer.0.attention.k_lin.weight gradient enabled: False
distilbert.transformer.layer.0.attention.k_lin.bias gradient enabled: False
distilbert.transformer.layer.0.attention.v_lin.linear.weight gradient enabled: False
distilbert.transformer.layer.0.attention.v_lin.linear.bias gradient enabled: False
distilbert.transformer.layer.0.attention.v_lin.lora._A gradient enabled: True
distilbert

We can see only LoRA layers have active gradients.

In [19]:
# count numbber of parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params}")

# count number of trainable parameters
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {count_trainable_params(model)}")

Number of parameters: 67102466
Number of trainable parameters: 147456


# Loading the Dataset

In [73]:
import os
import time
import urllib.request
import sys
import tarfile
from tqdm.auto import tqdm
from packaging import version
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

import lightning as pl
import torchmetrics
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

import pandas as pd
import torch

import warnings
warnings.filterwarnings("ignore")

In [74]:
start_time = time.time()

In [75]:
def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = start_time
        return
    duration = time.time() - start_time
    progress_size = int(count * block_size)
    speed = progress_size / (1024.0**2 * duration)
    percent = count * block_size * 100.0 / total_size
    sys.stdout.write(f"\r{int(percent)} | {progress_size / (1024.**2):.2f} MB "
                     f"| {speed:.2f} MB/s | {duration:.2f} sec elapsed")
    sys.stdout.flush()

In [76]:
def download_dataset():
    source = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    target = "./data/aclImdb_v1.tar.gz"

    if os.path.exists(target):
        os.remove(target)

    if not os.path.isdir("./data/aclImdb") and not os.path.isfile("./data/aclImdb_v1.tar.gz"):
        print("Downloading...")
        urllib.request.urlretrieve(source, target, reporthook=reporthook)

    if not os.path.isdir("./data/aclImdb"):
        print("\nExtracting...")
        # when only the tar file is present
        with tarfile.open(target, "r:gz") as tar:
            tar.extractall()

In [80]:
def load_dataset2dataframe():
    basepath = "./data/aclImdb"
    labels = {'pos': 1, 'neg': 0}

    df = pd.DataFrame()

    with tqdm(total=50000) as progress_bar:
        for subset in ("train", "test"):
            for label in ("pos", "neg"):
                path = os.path.join(basepath, subset, label)
                for file in sorted(os.listdir(path)):
                    with open(os.path.join(path, file), 'r', encoding="utf-8") as infile:
                        txt = infile.read()

                    if version.parse(pd.__version__) >= version.parse("1.3.2"):
                        x = pd.DataFrame(
                            [[txt, labels[label]]], columns=["review", "sentiment"])
                        df = pd.concat([df, x], ignore_index=False)
                    else:
                        df = df.append([[txt, labels[label]]], ignore_index=True)

                    progress_bar.update(1)  # update progress bar 1 file at a time

    df.columns = ["text", "label"]

    np.random.seed(0)
    df = df.reindex(np.random.permutation(df.index))   # shuffle the dataset

    print("Dataset shape:", df.shape)
    print("Dataset columns:", df.columns)
    print("Dataset head:", df.head())
    print("Class distribution:", np.bincount(df["label"].values))

    return df


In [81]:
def partition_dataset(df):
    df_shuffled = df.sample(frac=1, random_state=1).reset_index()

    train_size = 35_000

    train_df = df_shuffled.iloc[:train_size]
    val_df = df_shuffled.iloc[train_size:40_000]
    test_df = df_shuffled.iloc[40_000:]

    if not os.path.exists("./data"):
        print("Creating data folder...")
        os.makedirs("./data")

    train_df.to_csv(os.path.join("./data", "train.csv"), index=False, encoding="utf-8")
    val_df.to_csv(os.path.join("./data", "validation.csv"), index=False, encoding="utf-8")
    test_df.to_csv(os.path.join("./data", "test.csv"), index=False, encoding="utf-8")

In [82]:
def get_dataset():
    files = ("train.csv", "validation.csv", "test.csv")
    downloaded = True

    for file in files:
        if not os.path.exists(file):
            downloaded = False

    if downloaded is False:
        download_dataset()
        df = load_dataset2dataframe()
        partition_dataset(df)

    train_data = pd.read_csv(os.path.join("data", "train.csv"))
    val_data = pd.read_csv(os.path.join("data", "validation.csv"))
    test_data = pd.read_csv(os.path.join("data", "test.csv"))

    return train_data, val_data, test_data

In [83]:
def tokenization():
    imdb_dataset = load_dataset(
        "csv",
        data_files={
            "train": os.path.join("data", "train.csv"),
            "validation": os.path.join("data", "validation.csv"),
            "test": os.path.join("data", "test.csv"),
        },
    )
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    def tokenize_text(batch):
        return tokenizer(batch["text"], truncation=True, padding=True)

    imdb_tokenized = imdb_dataset.map(tokenize_text, batched=True, batch_size=None)
    imdb_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    return imdb_tokenized

In [84]:
class IMDBDataset(Dataset):
    def __init__(self, dataset_dict, partition_key="train"):
        self.partition = dataset_dict[partition_key]

    def __getitem__(self, idx):
        return self.partition[idx]

    def __len__(self):
        return self.partition.num_rows

In [85]:
class IMDBDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=12, num_workers=0):
        super().__init__()
        self.imdb_tokenized = None
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.batch_size = batch_size
        self.num_workers = num_workers

    def prepare_data(self):
        get_dataset()
        self.imdb_tokenized = tokenization()

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            self.train_dataset = IMDBDataset(self.imdb_tokenized, "train")
            self.val_dataset = IMDBDataset(self.imdb_tokenized, "validation")

        if stage == "test" or stage is None:
            self.test_dataset = IMDBDataset(self.imdb_tokenized, "test")

    def train_dataloader(self):
        return DataLoader(self.train_dataset,
                          batch_size=self.batch_size,
                          shuffle=True,
                          num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_dataset,
                          batch_size=self.batch_size,
                          num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset,
                          batch_size=self.batch_size,
                          num_workers=self.num_workers)

In [86]:
class IMDBSentimentClassifier(pl.LightningModule):
    def __init__(self, model, lr=2e-5):
        super().__init__()
        self.model = model
        self.lr = lr

        self.validation_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=2)
        self.test_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=2)

    def forward(self, input_ids, attention_mask, labels):
        return self.model(input_ids, attention_mask=attention_mask, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
        loss = outputs["loss"]
        self.log("train_loss", loss)
        return loss  # passed to optimizer for backpropagation step (training)

    def validation_step(self, batch, batch_idx):
        outputs = self(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
        logits = outputs["logits"]
        predicted_labels = torch.argmax(logits, dim=1)
        self.validation_accuracy(predicted_labels, batch["label"])
        self.log("val_accuracy", self.validation_accuracy, prog_bar=True)

    def test_step(self, batch, batch_idx):
        outputs = self(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
        logits = outputs["logits"]
        predicted_labels = torch.argmax(logits, dim=1)
        self.test_accuracy(predicted_labels, batch["label"])
        self.log("test_accuracy", self.test_accuracy, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        return optimizer

In [87]:
imdb_sentiment_classifier = IMDBSentimentClassifier(model)

In [88]:
callbacks = [
    ModelCheckpoint(
        monitor="val_accuracy",
        filename="imdb-sentiment-{epoch:02d}-{val_accuracy:.2f}",
        save_top_k=1,
        mode="max",
    )]

logger = CSVLogger("logs", name="imdb-sentiment")

In [89]:
start_time = time.time()

In [90]:
# define the trainer
trainer = pl.Trainer(
    max_epochs=1,
    callbacks=callbacks,
    accelerator="gpu",
    devices=1,
    logger=logger,
    log_every_n_steps=10
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [91]:
torch.set_float32_matmul_precision('medium')
trainer.fit(imdb_sentiment_classifier, imdb_data_module)


100%|██████████| 50000/50000 [00:23<00:00, 2086.73it/s]


Dataset shape: (50000, 2)
Dataset columns: Index(['text', 'label'], dtype='object')
Dataset head:                                                 text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
0  Homelessness (or Houselessness as George Carli...      1
0  Brilliant over-acting by Lesley Ann Warren. Be...      1
0  This is easily the most underrated film inn th...      1
0  This is not the typical Mel Brooks film. It wa...      1
Class distribution: [25000 25000]



Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 35000 examples [00:00, 156119.08 examples/s]

Generating validation split: 5000 examples [00:00, 162971.67 examples/s]

Generating test split: 10000 examples [00:00, 162392.42 examples/s]

Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Epoch 0:   5%|▌         | 147/2917 [04:32<1:25:38,  0.54it/s, v_num=0]



Map: 100%|██████████| 10000/10000 [00:04<00:00, 2280.10 examples/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                | Type                                | Params | Mode 
------------------------------------------------------------------------------------
0 | model               | DistilBertForSequenceClassification | 67.1 M | eval 
1 | validation_accuracy | MulticlassAccuracy                  | 0      | train
2 | test_accuracy       | MulticlassAccuracy                  | 0      | train
------------------------------------------------------------------------------------
147 K     Trainable params
67.0 M    Non-trainable params
67.1 M    Total params
268.410   Total estimated model params size (MB)
26        Modules in train mode
96        Modules in eval mode


Epoch 0:   6%|▌         | 161/2917 [00:33<09:27,  4.86it/s, v_num=1]       


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined